In [3]:
import os
import time
import torch
import torch.nn as nn
import numpy as np
import json
from PIL import Image
from torchvision import transforms
from scipy.cluster.hierarchy import fclusterdata
from modelSSD import SSD300
from detect import detect
from classification.src.models import model as cls_model
from utils import sort_bboxes, get_orientation
from NeurASP.neurasp import NeurASP

base_dir = os.getcwd()
images_folder= os.path.join(base_dir, "dataset", "images")
ssd_model_path= os.path.join(base_dir, "model", "tile_detection.pth")
color_model_path= os.path.join(base_dir, "classification", "color_last.pth")
number_model_path= os.path.join(base_dir, "classification", "number_last.pth")

output_file_json="neurasp_results.json"
checkpoint_file="checkpoint_neurasp.json" 

device=torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device used :",device)

def load_prev_checkpoint():
    if os.path.exists(checkpoint_file):
        with open(checkpoint_file,"r") as file:
            datas=json.load(file)
            done_images=set(datas.get("done_images",[]))
            results=datas.get("results",[])
            print(f"Checking checkpoints Resuming - {len(done_images)} images already processed,skipping them. ")
            return done_images,results
    print("Checkpoint not found - starting newly")
    return set(),[]

def saving_checkpoints(done_images,results):
    with open(checkpoint_file,"w") as write_file:
        json.dump({"done_images": list(done_images), "results" : results},write_file,indent=2)

class mymapping:
    def __init__(self):
        self.data={}

    def add(self, k, tensor):
        self.data[k]=tensor.unsqueeze(0)

    def __getitem__(self, k):
        return self.data[k]
    
    
class probnet(torch.nn.Module):
    def __init__(self, base_model, temperature=2.0):
        super().__init__()
        self.base=base_model
        self.temperature=temperature

    def forward(self,x):
        logits=self.base(x)/self.temperature
        probs=torch.softmax(logits,dim=1)
        return torch.clamp(probs,1e-6,1.0)

def clustering_tiles(boundingboxes,relative_threshold=1.5):
    if len(boundingboxes) == 0:
        return {}
    centres=np.array([[(box[0]+box[2])/2 , (box[1]+box[3])/2] for box in boundingboxes])
    average_size=(np.mean([box[2]-box[0] for box in boundingboxes]) +
                    np.mean([box[3]-box[1] for box in boundingboxes])) /2
    label=fclusterdata(centres, t=average_size*relative_threshold ,
                         criterion="distance")
    clusters={}
    for i,l in enumerate(label):
        clusters.setdefault(l-1,[]).append(boundingboxes[i])
    return clusters

def neurasp_program(tile_nm):

    domain="".join(f"tile({i}).\n" for i in tile_nm)
    nn_syntax=(
        "nn(color_net(1,X),  [black,blue,orange,red]):- tile(X).\n"
        "nn(number_net(1,X), [n1,n10,n11,n12,n13,n2,n3,n4,n5,n6,n7,n8,n9,n0]):- tile(X).\n"
    )

    bridge_rules= """
        tile_color(X,C)   :- color_net(_,X,C).
        tile_number(X,1)  :- number_net(_,X,n1).
        tile_number(X,2)  :- number_net(_,X,n2).
        tile_number(X,3)  :- number_net(_,X,n3).
        tile_number(X,4)  :- number_net(_,X,n4).
        tile_number(X,5)  :- number_net(_,X,n5).
        tile_number(X,6)  :- number_net(_,X,n6).
        tile_number(X,7)  :- number_net(_,X,n7).
        tile_number(X,8)  :- number_net(_,X,n8).
        tile_number(X,9)  :- number_net(_,X,n9).
        tile_number(X,10) :- number_net(_,X,n10).
        tile_number(X,11) :- number_net(_,X,n11).
        tile_number(X,12) :- number_net(_,X,n12).
        tile_number(X,13) :- number_net(_,X,n13).
        joker(X)          :- number_net(_,X,n0).
        """

    rules = """
        consecutive(1,2).  
        consecutive(2,3).  
        consecutive(3,4).
        consecutive(4,5).  
        consecutive(5,6).  
        consecutive(6,7).
        consecutive(7,8).  
        consecutive(8,9).  
        consecutive(9,10).
        consecutive(10,11). 
        consecutive(11,12). 
        consecutive(12,13).

        valid_set(A1,A2,A3) :-
            not joker(A1), 
            not joker(A2), 
            not joker(A3),
            tile_number(A1,N), 
            tile_number(A2,N), 
            tile_number(A3,N),
            tile_color(A1,C1), 
            tile_color(A2,C2), 
            tile_color(A3,C3),
            C1 != C2, 
            C1 != C3, 
            C2 != C3,
            A1 != A2, 
            A1 != A3, 
            A2 != A3.

        valid_set(A1,A2,A3) :-
            joker(A1), 
            not joker(A2), 
            not joker(A3),
            tile_number(A2,N), 
            tile_number(A3,N),
            tile_color(A2,C2), 
            tile_color(A3,C3),
            C2 != C3, 
            A2 != A3.

        valid_set(A1,A2,A3) :-
            joker(A2), 
            not joker(A1), 
            not joker(A3),
            tile_number(A1,N), 
            tile_number(A3,N),
            tile_color(A1,C1), 
            tile_color(A3,C3),
            C1 != C3, 
            A1 != A3.

        valid_set(A1,A2,A3) :-
            joker(A3), 
            not joker(A1), 
            not joker(A2),
            tile_number(A1,N), 
            tile_number(A2,N),
            tile_color(A1,C1), 
            tile_color(A2,C2),
            C1 != C2, 
            A1 != A2.

        valid_run(A1,A2,A3) :-
            not joker(A1), 
            not joker(A2), 
            not joker(A3),
            tile_color(A1,C), 
            tile_color(A2,C), 
            tile_color(A3,C),
            tile_number(A1,N1), 
            tile_number(A2,N2), 
            tile_number(A3,N3),
            consecutive(N1,N2), 
            consecutive(N2,N3),
            A1 != A2, 
            A1 != A3, 
            A2 != A3.

        valid_run(A1,A2,A3) :-
            joker(A1), 
            not joker(A2), 
            not joker(A3),
            tile_color(A2,C), 
            tile_color(A3,C),
            tile_number(A2,N2), 
            tile_number(A3,N3),
            consecutive(N2,N3), 
            A2 != A3.

        valid_run(A1,A2,A3) :-
            joker(A2), 
            not joker(A1), 
            not joker(A3),
            tile_color(A1,C), 
            tile_color(A3,C),
            tile_number(A1,N1), 
            tile_number(A3,N3),
            consecutive(N1,N2), 
            consecutive(N2,N3),
            A1 != A3.

        valid_run(A1,A2,A3) :-
            joker(A3), 
            not joker(A1), 
            not joker(A2),
            tile_color(A1,C), 
            tile_color(A2,C),
            tile_number(A1,N1), 
            tile_number(A2,N2),
            consecutive(N1,N2), 
            A1 != A2.
"""
    program_neurasp=domain + nn_syntax + bridge_rules + rules
    inference_rules=bridge_rules + rules
    return program_neurasp, inference_rules

def process_images(image_folder, ssd_model, color_net_prob, number_net_prob):

    transform=transforms.Compose([transforms.Resize((224, 224)),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], 
                             std=[0.229, 0.224, 0.225]),
      ])

    all_files=sorted([j for j in os.listdir(image_folder)
                        if j.lower().endswith((".jpg", ".png", ".jpeg"))])

    done_images, results_summary = load_prev_checkpoint()
    remaining=[f for f in all_files if f not in done_images]

    print(f"Total images:{len(all_files)}")
    print(f"Already done:{len(done_images)}")
    print(f"To process:{len(remaining)}\n")

    for image in remaining:
        image_path= os.path.join(image_folder, image)
        original_image= Image.open(image_path).convert("RGB")
        image_start= time.time()

        try:
            bounding_boxes, _, _ =detect(original_image,
                                          ssd_model,
                                          min_score=0.7,
                                          max_overlap=0.3,
                                          top_k=200)

            clusters=clustering_tiles(bounding_boxes)
            cluster_sorted = {}
            for i, boxes in clusters.items():
                orientation          = get_orientation(boxes)
                cluster_sorted[i] = sort_bboxes(boxes, orientation)

            tile_mapping=mymapping()

            for i, boxes in cluster_sorted.items():
                for j, bbox in enumerate(boxes):
                    name=f"t{i}_{j}"
                    tile_mapping.add(name, transform(original_image.crop(bbox)))

            nnMapping ={"color_net": color_net_prob, "number_net": number_net_prob}
            n_run=0
            n_group =0

            for i, boxes in cluster_sorted.items():
                names=[f"t{i}_{j}" for j in range(len(boxes))]
                for name in range(len(names) - 2):
                    triplet=[names[name], names[name+1], names[name+2]]
                    
                    program, inference = neurasp_program(triplet)
                    triplet_data={t: tile_mapping.data[t] for t in triplet}
                    
                    neurasp_obj=NeurASP(program, nnMapping, optimizers=None)
                    stable_mod=neurasp_obj.infer(
                        dataDic=triplet_data,
                        mvpp=inference
                    )

                    if stable_mod:
                        model= set(str(x) for x in stable_mod[0])
                        t1, t2, t3= triplet
                        if f"valid_run({t1},{t2},{t3})" in model:
                            n_run += 1
                            print(f"  valid_run ({t1},{t2},{t3})")
                        elif f"valid_set({t1},{t2},{t3})" in model:
                            n_group += 1
                            print(f"  valid_set ({t1},{t2},{t3})")
                        else:
                            print(f"  neither set or run ({t1},{t2},{t3})")

            img_time = time.time() - image_start
            print(f"NeurASP {image}: valid_run={n_run}  valid_set={n_group}  time={img_time:.3f}s")

            results_summary.append({
                "image":         image,
                "num_clusters":  len(cluster_sorted),
                "n_valid_run":   n_run,
                "n_valid_set": n_group,
                "confidence":    1.0 if (n_run + n_group) > 0 else 0.0,
                "time_seconds":  round(img_time, 4),
            })

        except Exception as e:
            img_time = time.time() - image_start
            print(f"[ERROR] {image} failed after {img_time:.2f}s: {e} — skipping.")
        done_images.add(image)
        saving_checkpoints(done_images, results_summary)

    return results_summary


if __name__ == "__main__":
    ssd_model=SSD300(2, device)
    ssd_model.load_state_dict(torch.load(ssd_model_path, map_location=device))
    ssd_model.eval()

    color_model=cls_model.get_model("resnet18", None, out_features=4)
    number_model=cls_model.get_model("resnet18", None, out_features=14)

    color_model.load_state_dict(torch.load(color_model_path,  map_location=device))
    number_model.load_state_dict(torch.load(number_model_path, map_location=device))

    color_model.eval()
    number_model.eval()

    color_net_prob =probnet(color_model)
    number_net_prob=probnet(number_model)

    total_start=time.time()
    results_summary= process_images(images_folder, ssd_model,
                                     color_net_prob, number_net_prob)
    total_time=time.time() - total_start

    full_total_time=round(sum(r["time_seconds"] for r in results_summary), 4)

    print(f"\nNeurASP total inference time: {full_total_time:.2f}s")

    output = {
        "total_time": round(full_total_time, 4),
        "results":    results_summary,
    }
    with open(output_file_json, "w") as f:
        json.dump(output, f, indent=2)
    print(f"Saved → {output_file_json}")
    if os.path.exists(checkpoint_file):
        os.remove(checkpoint_file)
        print(f"Deleted checkpoint ({checkpoint_file}) — all done!")

Device used : cpu
using pretrained weights

Loaded base model.

Checkpoint not found - starting newly
Total images:285
Already done:0
To process:285

Inside init
Insider infer
  valid_run (t1_0,t1_1,t1_2)
Inside init
Insider infer
  valid_run (t1_1,t1_2,t1_3)
Inside init
Insider infer
  valid_set (t0_0,t0_1,t0_2)
NeurASP 20220716_115940.jpg: valid_run=2  valid_set=1  time=2.125s
Inside init
Insider infer
  valid_run (t0_0,t0_1,t0_2)
Inside init
Insider infer
  valid_run (t0_1,t0_2,t0_3)
Inside init
Insider infer
  valid_set (t2_0,t2_1,t2_2)
Inside init
Insider infer
  valid_set (t1_0,t1_1,t1_2)
NeurASP 20220716_115959.jpg: valid_run=2  valid_set=2  time=2.143s
Inside init
Insider infer
  valid_set (t0_0,t0_1,t0_2)
Inside init
Insider infer
  valid_run (t1_0,t1_1,t1_2)
Inside init
Insider infer
  valid_run (t2_0,t2_1,t2_2)
NeurASP 20220716_120023.jpg: valid_run=2  valid_set=1  time=1.704s
Inside init
Insider infer
  valid_set (t0_0,t0_1,t0_2)
NeurASP 20220716_144734.jpg: valid_run=0  va